## 第11章　偏导数 —— 按住一个，动另一个

> 本章目标：将第 10 章的一元导数推广到多元——偏导数 ∂f/∂x 就是"把其他变量当常数，只对 x 求导"。掌握 `np.gradient` 在二维网格上批量计算偏导，并通过等高线图建立"梯度垂直于等高线"的核心几何直觉——这是第 12 章梯度下降的视觉基础。
> 前置知识：第 10 章（导数）、第 6 章（矩阵运算）、第 3 章（shape 与轴）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('env ready')



### 11.1　二元函数的偏导数 —— 按住一个，动另一个

损失函数的输入从来不是一个标量，而是几百万个权重参数。要同时对所有参数"求导"，你需要**偏导数（Partial Derivative）**——它的操作哲学出奇简单：**把其他变量全当成常数，只对目标变量用第 10 章的求导规则。**

符号从 `d` 变成 `∂`（读作"round d"或"partial d"），提醒你这是一个多元世界。对于 f(x, y) = x² + y²：

- ∂f/∂x = 2x —— 把 y² 当常数，它的导数是 0
- ∂f/∂y = 2y —— 把 x² 当常数，它的导数是 0

直觉：你站在一个山谷里（f 代表高度）。∂f/∂x 告诉你"只往东走一步，高度怎么变"；∂f/∂y 告诉你"只往北走一步，高度怎么变"。两个方向的偏导数合在一起——[∂f/∂x, ∂f/∂y]——就是**梯度**（第 12 章），它指向"最陡上坡"的方向。

📐 **定义　偏导数（Partial Derivative）**：∂f/∂x = lim(h→0) [f(x+h, y) − f(x, y)] / h。**把除目标变量外的所有变量当常数**，然后按一元函数求导。符号 ∂ 区别于 d，强调函数的多元性。

💻 **代码　数值偏导数——用中心差分分别扰动 x 和 y**




In [ ]:
import numpy as np

def f(x, y):
    return x**2 + y**2

def partial_x(f, x, y, h=1e-5):
    """只扰动 x，保持 y 不变"""
    return (f(x + h, y) - f(x - h, y)) / (2 * h)

def partial_y(f, x, y, h=1e-5):
    """只扰动 y，保持 x 不变"""
    return (f(x, y + h) - f(x, y - h)) / (2 * h)

# 在点 (3, 4) 处验证
x0, y0 = 3.0, 4.0
px_num = partial_x(f, x0, y0)
py_num = partial_y(f, x0, y0)

print(f"在 ({x0}, {y0}) 处:")
print(f"  ∂f/∂x: 数值={px_num:.6f}  解析=2×{x0}={2*x0:.1f}  ✓")
print(f"  ∂f/∂y: 数值={py_num:.6f}  解析=2×{y0}={2*y0:.1f}  ✓")

# 验证偏导数的核心直觉：∂f/∂x 的解析值只依赖 x，与 y 无关
for y_test in [0.0, 5.0, 10.0]:
    px = partial_x(f, x0, y_test)
    print(f"  ∂f/∂x at (3, {y_test:4.1f}) = {px:.1f} ← 与 y 无关！(因为 f_x=2x)")




> **关键洞察**：偏导数的计算本质是**逐维度的独立测量**——把多维问题分解为多个一维问题。神经网络中损失函数对权重矩阵 W 的梯度是一个和 W 同 shape 的张量，其中每个元素 `∂L/∂W[i,j]` 都是"固定其他所有权重参数，只动 W[i,j]"的偏导数。第 14 章的反向传播就是用链式法则高效计算所有这些偏导数。

🔗 **AI 连接**：PyTorch 的 `loss.backward()` 一次性算出所有参数的偏导数（存储在 `param.grad` 中），每个参数的 `.grad` 形状和参数本身一致——这就是"把偏导数打包成梯度张量"的工程实现（第 14 章揭晓）。

---

### 11.2　np.gradient —— 在整个网格上一次性求所有点的偏导

手工写的 `partial_x` 每次只能算一个点。AI 常需要在整片区域上看到导数全貌——比如画出整个损失曲面的"坡度图"。`np.gradient` 就是干这个的：输入二维网格上的函数值 Z，一次性返回所有点的 ∂Z/∂x 和 ∂Z/∂y。

工作原理：对每个内部点用中心差分，对边缘点用单侧差分——和你 11.1 节的手工逻辑完全一致，只是批量化了。

📐 **定义　np.gradient**：`np.gradient(Z, y, x)` 返回 `(∂Z/∂y, ∂Z/∂x)`。**注意返回顺序：第一个轴（y 方向）在前！** 这是因为 NumPy 的 axis=0 是"行"方向（对应 y），axis=1 是"列"方向（对应 x）。

💻 **代码　np.gradient 批量求偏导 + 验证**




In [ ]:
import numpy as np

# 生成二维网格
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)          # X: (50,50), Y: (50,50)
Z = X**2 + Y**2                    # f(x,y) 在网格上的值

# np.gradient — 返回 (∂Z/∂y, ∂Z/∂x)  注意顺序！
dZ_dy, dZ_dx = np.gradient(Z, y, x)

print(f"Z.shape: {Z.shape}")
print(f"∂Z/∂x.shape: {dZ_dx.shape}  ∂Z/∂y.shape: {dZ_dy.shape}\n")

# 验证几个点
# 原点附近 (x≈0, y≈0): ∂Z/∂x = 2x ≈ 0
center_i = 25  # X[25,25] ≈ 0
print(f"在原点附近 (网格索引 25,25):")
print(f"  np.gradient ∂Z/∂x = {dZ_dx[center_i, center_i]:.4f}  (期望 ≈ 0)")
print(f"  np.gradient ∂Z/∂y = {dZ_dy[center_i, center_i]:.4f}  (期望 ≈ 0)")

# 角落 (x≈3, y≈3): ∂Z/∂x = 2x ≈ 6, ∂Z/∂y = 2y ≈ 6
print(f"在角落 (x≈3, y≈3):")
print(f"  np.gradient ∂Z/∂x = {dZ_dx[-1, -1]:.4f}  (期望 ≈ 6)")
print(f"  np.gradient ∂Z/∂y = {dZ_dy[-1, -1]:.4f}  (期望 ≈ 6)")
print("\n关键提醒: np.gradient 返回 (dZ/dy, dZ/dx)——y在前，x在后！")




> **关键洞察**：`np.gradient` 的返回顺序 `(dZ/dy, dZ/dx)` 是 NumPy 初学者最容易栽的坑——因为 axis=0 对应 y 方向（行），axis=1 对应 x 方向（列）。记忆技巧：**"gradient 返回 tuple，第一个元素对应第一个轴"**。这个顺序在第 12 章画梯度场时会反复用到。

🔗 **AI 连接**：`np.gradient` 在第 12 章用于画损失曲面的梯度箭头图，直观展示"梯度指向最陡上升方向"。PyTorch 中 `torch.gradient` 有相同的行为。

---

### 11.3　等高线图 —— 梯度垂直于等高线，指向"最陡上坡"

现在来到本章最重要的几何直觉。

想象你站在一座山上（f(x,y) 代表海拔）。等高线是把海拔相同的点连成的闭合曲线——同一圈上，高度不变。如果你站在等高线上环顾四周，**哪个方向上山最快？** 答案是：**垂直于等高线，直接往外（或往里）走。**

这就是梯度方向的几何含义：∇f = [∂f/∂x, ∂f/∂y] 指向**与等高线垂直**的方向，且指向**函数值增长最快**的方向。−∇f 指向最陡下降方向——这就是梯度下降走的方向（第 12 章）。

**为什么偏导垂直于等高线？** 因为沿等高线方向走一步，高度不变 → 方向导数 = 0 → 梯度在该方向的投影 = 0 → 梯度垂直于等高线。一个自然推论：如果你已经站在谷底（最小值），梯度 = 0——这正是优化的目标。

💻 **代码　等高线 + 偏导箭头：亲眼看到"梯度指向圈外"**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 构建 f(x,y) = x² + 2y²（椭圆形等高线，比正圆形更有意思）
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Z = X**2 + 2 * Y**2

# 用 np.gradient 求偏导
dZ_dy, dZ_dx = np.gradient(Z, y, x)

fig, ax = plt.subplots(figsize=(8, 7))

# 等高线
cs = ax.contour(X, Y, Z, levels=15, cmap='viridis', alpha=0.6)
ax.clabel(cs, fontsize=7)

# 画负梯度箭头（指向最陡下降方向——走向谷底）
skip = 3  # 每隔 3 个点画一个箭头，避免过于密集
ax.quiver(X[::skip, ::skip], Y[::skip, ::skip],
          -dZ_dx[::skip, ::skip], -dZ_dy[::skip, ::skip],
          color='red', alpha=0.7, scale=80, width=0.004,
          label='-∇f (最陡下降方向)')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('−∇f 箭头总是垂直于等高线，指向谷底 (最小值)')
ax.legend(); ax.set_aspect('equal'); plt.show()

print("观察: 每一个红色箭头都恰好垂直于穿过它的等高线")
print("      箭头从外向内指向谷底(x=0, y=0)——这就是梯度下降的几何直觉")
print("      ∇f 垂直于等高线 → −∇f 也垂直于等高线 → 梯度下降'直插'谷底")




> **关键洞察**：红色的负梯度箭头——**每一个都精确地垂直于穿过它的等高线，从高海拔直指低海拔。** 这张图就是整个梯度下降算法（第 12 章）的视觉核心：每一步更新 `θ -= lr * ∇L`，本质上就是在损失曲面上沿着"垂直等高线且指向谷底"的方向迈一小步。

🔗 **AI 连接**：第 12 章将把"梯度垂直于等高线"这个几何直觉翻译成参数更新公式 `θ_new = θ_old − lr * ∇L(θ)`，并在真正的损失函数上沿负梯度方向走几步，亲眼看到 loss 下降。第 14 章将揭秘 PyTorch 如何自动算出这上亿个偏导数。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）偏导数 ∂f/∂x 和一元导数 df/dx 在符号上有什么区别（∂ vs d）？这背后的数学含义是什么？
2. （概念）"梯度向量垂直于等高线"——用"站在山上环顾四周，哪个方向上山最快"的比喻来解释这句话。
3. （代码）对 f(x,y) = sin(x)·cos(y)，在 [−π, π] × [−π, π] 的二维网格上用 `np.gradient` 求偏导，画出等高线 + 负梯度箭头图。指出图中的极大值点、极小值点和鞍点，观察这些点上的梯度箭头行为。
---


> 🔗 **章末钩子**：偏导数让你能逐个维度分析变化率。把它们打包成一个向量——[∂f/∂x, ∂f/∂y]——就是深度学习中最核心的概念之一。
> 预览：下一章——**梯度向量**，数学工具的首次融合：从偏导数到梯度下降。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
